In [1]:
import matplotlib.pyplot as plt 

In [2]:
from Bio import Entrez, SeqIO
import time
import os 


In [3]:
"""
Step 1: Fetch PDCD1 (PD-1) and CD274 (PD-L1) mRNA/CDS sequences
across 5 species using NCBI Entrez.

Biological question this project answers:
    How conserved are the receptor (PD-1) and ligand (PD-L1) genes
    of the immune checkpoint axis across mammals and a non-mammalian
    outgroup (zebrafish)? Does conservation pattern differ between
    the receptor and its ligand?

Species chosen:
    - Homo sapiens        (human)
    - Mus musculus        (mouse)
    - Pan troglodytes     (chimpanzee)
    - Rattus norvegicus   (rat)
    - Gallus gallus       (chicken) -- non-mammalian outgroup, tests how
      far back this checkpoint axis is conserved beyond mammals

    Note: zebrafish (Danio rerio) was originally considered as the
    outgroup, but PDCD1/CD274 are not yet reliably annotated under
    matching gene symbols in NCBI for zebrafish -- PD-1 was only very
    recently identified outside tetrapods (2025 literature), so it
    isn't consistently catalogued yet. Chicken was substituted instead:
    still evolutionarily distant from mammals, but well-annotated.

Requirements:
    pip install biopython

IMPORTANT: NCBI requires an email address with every request (their
usage policy, not optional). Replace YOUR_EMAIL_HERE below. If you
get frequent timeouts, get a free NCBI API key and set Entrez.api_key
too -- raises your rate limit from 3/sec to 10/sec.
"""

from Bio import Entrez, SeqIO
import time
import os

# ---- CONFIG ----
Entrez.email = "YOUR_EMAIL_HERE"      # required by NCBI
# Entrez.api_key = "YOUR_NCBI_API_KEY"  # optional, uncomment if you have one

GENES = ["PDCD1", "CD274"]

SPECIES = {
    "human":     "Homo sapiens",
    "mouse":     "Mus musculus",
    "chimp":     "Pan troglodytes",
    "rat":       "Rattus norvegicus",
    "chicken":   "Gallus gallus",
}

OUTPUT_ROOT = "sequences"


def search_refseq_mrna(gene_symbol, organism, debug=False):
    """
    Search NCBI nucleotide db for a RefSeq mRNA record matching
    the gene symbol + organism. Returns the top matching UID, or
    None if nothing found.

    Uses the documented Entrez "Properties" tags for nucleotide db
    (srcdb_refseq = source database is RefSeq, biomol_mrna = molecule
    type is mRNA) instead of "[Filter]", which is not a reliable tag
    for this database and was silently returning zero hits.

    IMPORTANT: when a gene has multiple transcript variants (common --
    e.g. CD274 has 4+), NCBI's relevance ranking does NOT reliably put
    the canonical/full-length variant first. We fix this by fetching
    summaries for ALL candidates and picking the LONGEST NM_ (curated)
    record, since the canonical/full-length transcript is typically
    the longest one. This is a heuristic, not a guarantee -- worth a
    manual spot-check against known protein length for any gene this
    matters for.
    """
    query = (
        f"{gene_symbol}[Gene Name] AND {organism}[Organism] "
        f"AND srcdb_refseq[Properties] AND biomol_mrna[Properties]"
    )
    handle = Entrez.esearch(db="nucleotide", term=query, retmax=20, sort="relevance")
    record = Entrez.read(handle)
    handle.close()

    if debug:
        print(f"      [debug] query: {query}")
        print(f"      [debug] raw result: {record}")

    ids = record.get("IdList", [])

    if not ids:
        # Fallback: retry without the strict properties filters, in case
        # the properties tags aren't matching for this particular record.
        fallback_query = f"{gene_symbol}[Gene Name] AND {organism}[Organism]"
        handle = Entrez.esearch(db="nucleotide", term=fallback_query, retmax=20, sort="relevance")
        record = Entrez.read(handle)
        handle.close()

        if debug:
            print(f"      [debug] fallback query: {fallback_query}")
            print(f"      [debug] fallback result count: {record.get('Count')}")

        ids = record.get("IdList", [])
        if not ids:
            return None

    # We have at least one candidate. Fetch summaries for ALL of them so
    # we can pick the best one, not just whatever NCBI ranked first.
    handle = Entrez.esummary(db="nucleotide", id=",".join(ids))
    summaries = Entrez.read(handle)
    handle.close()

    if debug:
        for s in summaries:
            print(f"      [debug] candidate: {s.get('Caption')}  "
                  f"length={s.get('Length')}  title={s.get('Title')}")

    # Prefer curated NM_ records over predicted XM_ records. Among
    # whichever group we have, pick the longest -- canonical/full-length
    # transcripts are typically the longest available.
    nm_candidates = [s for s in summaries if s.get("Caption", "").startswith("NM_")]
    pool = nm_candidates if nm_candidates else summaries

    best = max(pool, key=lambda s: int(s.get("Length", 0)))

    if debug:
        print(f"      [debug] selected: {best.get('Caption')} (length={best.get('Length')})")

    return best["Id"]


def fetch_fasta(uid):
    """Fetch a single record as FASTA text given its UID."""
    handle = Entrez.efetch(db="nucleotide", id=uid, rettype="fasta", retmode="text")
    data = handle.read()
    handle.close()
    return data


def main():
    for gene in GENES:
        gene_dir = os.path.join(OUTPUT_ROOT, gene)
        os.makedirs(gene_dir, exist_ok=True)

        print(f"\n=== Fetching {gene} ===")
        for species_key, organism_name in SPECIES.items():
            print(f"  {species_key} ({organism_name})...", end=" ")

            uid = search_refseq_mrna(gene, organism_name, debug=True)
            if uid is None:
                print("NOT FOUND -- may need manual accession lookup")
                continue

            fasta_text = fetch_fasta(uid)

            out_path = os.path.join(gene_dir, f"{species_key}.fasta")
            with open(out_path, "w") as f:
                f.write(fasta_text)

            # quick sanity check on what we actually pulled down
            header = fasta_text.split("\n")[0]
            print(f"OK -> {out_path}")
            print(f"      header: {header}")

            # be polite to NCBI's rate limit (3 req/sec without API key,
            # each gene/species pair = 2 requests, so this is safely under)
            time.sleep(0.4)

    print("\nDone. Check the 'sequences/' folder -- one FASTA per species per gene.")
    print("Next step: parse these with SeqIO and confirm sequence lengths/IDs")
    print("look sane before moving into alignment.")


if __name__ == "__main__":
    main()


=== Fetching PDCD1 ===
  human (Homo sapiens)...       [debug] query: PDCD1[Gene Name] AND Homo sapiens[Organism] AND srcdb_refseq[Properties] AND biomol_mrna[Properties]
      [debug] raw result: {'Count': '4', 'RetMax': '4', 'RetStart': '0', 'IdList': ['1519242236', '2462574096', '2462489320', '2217328688'], 'TranslationSet': [{'From': 'Homo sapiens[Organism]', 'To': '"Homo sapiens"[Organism]'}], 'TranslationStack': [{'Term': 'PDCD1[Gene Name]', 'Field': 'Gene Name', 'Count': '1885', 'Explode': 'N'}, {'Term': '"Homo sapiens"[Organism]', 'Field': 'Organism', 'Count': '29841992', 'Explode': 'Y'}, 'AND', {'Term': 'srcdb_refseq[Properties]', 'Field': 'Properties', 'Count': '149442002', 'Explode': 'N'}, 'AND', {'Term': 'biomol_mrna[Properties]', 'Field': 'Properties', 'Count': '251678236', 'Explode': 'N'}, 'AND'], 'QueryTranslation': 'PDCD1[Gene Name] AND "Homo sapiens"[Organism] AND srcdb_refseq[Properties] AND biomol_mrna[Properties]'}
      [debug] candidate: NM_005018  length=Integer

In [4]:
#Step 2: SeqIO — reading the FASTA files back in

from Bio import SeqIO
import os 


GENES = ["PDCD1", "CD274"]
SEQUENCES_ROOT = "sequences"

In [5]:
def load_and_check_gene(gene):
    """
    Parse every FASTA file for a given gene and print a sanity check 
    summary table. Returns a dict of {species: SeqRecord} for use in later
    steps (e.g. writing a combined multi-FASTA for alignment).
    """

    gene_dir = os.path.join(SEQUENCES_ROOT, gene)
    records = {}

    print(f"\n ==={gene}===")
    print(f"{'species':<12} {'lenght (bp)': <12} {'accession': <18} description")
    print("-" * 90)

    for filename in sorted(os.listdir(gene_dir)):
        if not filename.endswith(".fasta"):
            continue

        species = filename.replace(".fasta", "")
        filepath = os.path.join(gene_dir, filename)

        record = SeqIO.read(filepath, "fasta")
        records[species] = record

        print(f"{species:<12} {len(record.seq):<12} {record.id:<18} {record.description}")

    return records 




In [6]:
def main():
    all_records = {}
    for gene in GENES:
        all_records[gene] = load_and_check_gene(gene)

    

    print("-----What to eyeball-----")
    print("1. Lenghts should be in a plausible range for mRNA")
    print("   (a few hundred to a few thousand bp -- if something is)")
    print("   10x longer/shorter than its siblings, investigate before")
    print("   moving on).")
    print("2. Description should mention the right gene and species--")
    print("   catches cases where a search grabbed the wrong record.")
    print("3. Accession prefix: NM_ = curated RefSeq, XM_ = computationally")
    print("   predicted.")

    return all_records

print(">>> Step 2 script starting...")
all_records = main()
print(">>> Step 2 script finished.")

>>> Step 2 script starting...

 ===PDCD1===
species      lenght (bp)  accession          description
------------------------------------------------------------------------------------------
chicken      1432         XM_040678631.2     XM_040678631.2 PREDICTED: Gallus gallus programmed cell death 1 (PDCD1), mRNA
chimp        2130         XM_016950951.4     XM_016950951.4 PREDICTED: Pan troglodytes programmed cell death 1 (PDCD1), transcript variant X4, mRNA
human        2097         NM_005018.3        NM_005018.3 Homo sapiens programmed cell death 1 (PDCD1), mRNA
mouse        1933         NM_008798.3        NM_008798.3 Mus musculus programmed cell death 1 (Pdcd1), mRNA
rat          1927         NM_001106927.1     NM_001106927.1 Rattus norvegicus programmed cell death 1 (Pdcd1), mRNA

 ===CD274===
species      lenght (bp)  accession          description
------------------------------------------------------------------------------------------
chicken      2164         XM_046935715.1   

In [6]:

from Bio import SeqIO
import subprocess
import os

GENES = ["PDCD1", "CD274"]
SEQUENCES_ROOT = "sequences"
ALIGNED_ROOT = "aligned"
MUSCLE_PATH = "./muscle3"   # adjust if your binary is elsewhere/named differently


def combine_fastas(gene):
    """
    Read every per-species FASTA for a gene and write them all into
    one combined multi-FASTA file. Returns the path to that file.
    """
    gene_dir = os.path.join(SEQUENCES_ROOT, gene)
    combined_path = os.path.join(gene_dir, f"{gene}_combined.fasta")

    records = []
    all_files = sorted(os.listdir(gene_dir))
    print(f"  [debug] files in {gene_dir}: {all_files}")

    for filename in all_files:
        if not filename.endswith(".fasta") or filename.endswith("_combined.fasta"):
            continue
        filepath = os.path.join(gene_dir, filename)
        record = SeqIO.read(filepath, "fasta")

        
        species = filename.replace(".fasta", "")
        record.id = species
        record.description = ""  # clear so it doesn't clutter the header

        records.append(record)

    # SeqIO.write() takes a list of records and writes them all to one file
    SeqIO.write(records, combined_path, "fasta")
    print(f"  Combined {len(records)} sequences -> {combined_path}")
    return combined_path


def run_muscle(input_path, output_path):
    """
    Call the muscle3 binary as an external program from Python.

    subprocess.run() is how Python launches and controls a separate
    program (as if you'd typed the command yourself in Terminal).
    We're NOT importing muscle as a Python library -- it's not one --
    we're literally running it like a command-line tool and waiting
    for it to finish.
    """
    command = [
        MUSCLE_PATH,
        "-in", input_path,      # v3 syntax: -in (v5 would use -align instead)
        "-out", output_path,
    ]

    print(f"  Running: {' '.join(command)}")
    result = subprocess.run(command, capture_output=True, text=True)

    if result.returncode != 0:
        # returncode != 0 means the program hit an error
        print("  MUSCLE reported an error:")
        print(result.stderr)
        return False

    print(f"  Alignment written -> {output_path}")
    return True


def main():
    print(">>> Step 3 script starting...")
    os.makedirs(ALIGNED_ROOT, exist_ok=True)

    for gene in GENES:
        print(f"\n=== {gene} ===")
        combined_path = combine_fastas(gene)

        aligned_path = os.path.join(ALIGNED_ROOT, f"{gene}_aligned.fasta")
        success = run_muscle(combined_path, aligned_path)

        if not success:
            print(f"  Skipping {gene} due to MUSCLE error -- check message above.")

    print("\n>>> Step 3 script finished.")
    print("Check the 'aligned/' folder for the alignment output files.")
    print("Next: open one in a text editor and look at how the '-' gap")
    print("characters line up the sequences -- that's the alignment working.")


if __name__ == "__main__":
    main()

>>> Step 3 script starting...

=== PDCD1 ===
  [debug] files in sequences/PDCD1: ['PDCD1_combined.fasta', 'chicken.fasta', 'chimp.fasta', 'human.fasta', 'mouse.fasta', 'rat.fasta']
  Combined 5 sequences -> sequences/PDCD1/PDCD1_combined.fasta
  Running: ./muscle3 -in sequences/PDCD1/PDCD1_combined.fasta -out aligned/PDCD1_aligned.fasta
  Alignment written -> aligned/PDCD1_aligned.fasta

=== CD274 ===
  [debug] files in sequences/CD274: ['CD274_combined.fasta', 'chicken.fasta', 'chimp.fasta', 'human.fasta', 'mouse.fasta', 'rat.fasta']
  Combined 5 sequences -> sequences/CD274/CD274_combined.fasta
  Running: ./muscle3 -in sequences/CD274/CD274_combined.fasta -out aligned/CD274_aligned.fasta
  Alignment written -> aligned/CD274_aligned.fasta

>>> Step 3 script finished.
Check the 'aligned/' folder for the alignment output files.
Next: open one in a text editor and look at how the '-' gap
characters line up the sequences -- that's the alignment working.


In [9]:


from Bio import AlignIO

GENES = ["PDCD1", "CD274"]
ALIGNED_ROOT = "aligned"

print(">>> Alignment check starting...")

for gene in GENES:
    path = f"{ALIGNED_ROOT}/{gene}_aligned.fasta"
    alignment = AlignIO.read(path, "fasta")

    print(f"\n==={gene}===")
    print(f"Number of sequences: {len(alignment)}")
    print(f"Alignment lenght (with gaps): {alignment.get_alignment_length()}")
    print()

    for record in alignment:
        gap_count = record.seq.count("-")
        print(f" {record.id:<10} gaps: {gap_count:<6}"
              f"first 60 cols: {record.seq[:60]}")



print("\n>>> What to eyeball:")
print("1. Every sequence should show the SAME 'alignment length' --")
print("   that's the whole point of alignment, everything gets padded")
print("   to match.")
print("2. Gap counts that are wildly different from siblings can mean")
print("   a real indel (insertion/deletion) OR a bad/truncated sequence")
print("   -- worth a second look, not necessarily a problem.")
print("3. In the 'first 60 cols' preview, '-' characters are gaps MUSCLE")
print("   inserted so everything lines up in the same columns.")


>>> Alignment check starting...

===PDCD1===
Number of sequences: 5
Alignment lenght (with gaps): 2302

 chicken    gaps: 870   first 60 cols: -----------------------------------GGGCACAGGCACAGGGCACAGCAGG
 chimp      gaps: 172   first 60 cols: CTCCCTTCAGCCTGACCTGGGACAGTTTCCCTTCCGCTCACCTCCGCCTGA-GCAGTGGA
 human      gaps: 205   first 60 cols: -----------------------------------GCTCACCTCCGCCTGA-GCAGTGGA
 mouse      gaps: 369   first 60 cols: ------------------------------------------------TGA-GCAGCGGG
 rat        gaps: 375   first 60 cols: ------------------------------------------------TGA-GCAGCGGA

===CD274===
Number of sequences: 5
Alignment lenght (with gaps): 4675

 chicken    gaps: 2511  first 60 cols: GGAAGCGGTTGAACTTCCAGCCCTCTGACGCCAGCCCACGTCCACTCGTGTTCGGGAATT
 chimp      gaps: 1146  first 60 cols: ------------------------------------------------------GCAGCT
 human      gaps: 1383  first 60 cols: ------------------------------------------------------------
 mouse      gaps: 1022  

In [7]:
#step 4: BUILDING A PHYLOGENETIC TREE FROM EACH GENES ALIGNMENT

from Bio import AlignIO, Phylo
from Bio.Phylo.TreeConstruction import DistanceTreeConstructor, DistanceCalculator

GENES = ["PDCD1", "CD274"]
ALIGNED_ROOT = "aligned"
TREES_ROOT = "trees"

import os 
os.makedirs(TREES_ROOT, exist_ok=True)

print(">>> Step 4 script starting...")

for gene in GENES:
    print(f"\n==={gene}===")

    alignment = AlignIO.read(f"{ALIGNED_ROOT}/{gene}_aligned.fasta", "fasta")

    calculator = DistanceCalculator("identity")
    distance_matrix = calculator.get_distance(alignment)
    print("Distance matrix:")
    print(distance_matrix)


    constructor = DistanceTreeConstructor()
    tree = constructor.nj(distance_matrix)


    tree_path = f"{TREES_ROOT}/{gene}_tree.nwk"
    Phylo.write(tree, tree_path, "newick")
    print(f"Tree saved ---> {tree_path}")


    print("/nAscii tree:")
    Phylo.draw_ascii(tree)

print("\n>>> Step 4 script finished.")


>>> Step 4 script starting...

===PDCD1===
Distance matrix:
chicken 0.000000
chimp   0.619896    0.000000
human   0.604692    0.033015    0.000000
mouse   0.570374    0.450043    0.436577    0.000000
rat 0.583840    0.448306    0.434405    0.140313    0.000000
    chicken chimp   human   mouse   rat
Tree saved ---> trees/PDCD1_tree.nwk
/nAscii tree:
                    __________________________________________________ chicken
  _________________|
 |                 |                             , human
 |                 |_____________________________|
_|                                               |__ chimp
 |
 |________ mouse
 |
 |________ rat


===CD274===
Distance matrix:
chicken 0.000000
chimp   0.654143    0.000000
human   0.458398    0.225768    0.000000
mouse   0.467490    0.562189    0.371933    0.000000
rat 0.473838    0.569223    0.371762    0.172585    0.000000
    chicken chimp   human   mouse   rat
Tree saved ---> trees/CD274_tree.nwk
/nAscii tree:
                 ___